# Scale factor computation

The purpose of this notebook is to estimate the scale vector to use to make sure the latent space has unit variance.

In [1]:
import os
from pathlib import Path

import torch
from tqdm import tqdm

In [2]:
# Find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# Set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: C:\Users\couch\Documents\robust-radiotherapy-planning


In [3]:
train_folder = Path("RESULTS/MAISI_TRAINING/latents/train")
val_folder = Path("RESULTS/MAISI_TRAINING/latents/val")
train_files = list(train_folder.glob("*.pt"))
val_files = list(val_folder.glob("*.pt"))
latent_files = train_files + val_files

print(f"Found {len(latent_files)} large latent files.")

Found 1613 large latent files.


In [4]:
total_elements = 0
running_sum = 0.0
running_sum_sq = 0.0

for file_path in tqdm(train_files, desc="Processing latents"):
    # Load one file at a time
    latent = torch.load(file_path, map_location="cpu").to(torch.float32)

    # Accumulate element count, sum, and sum of squares
    total_elements += latent.numel()
    running_sum += torch.sum(latent).item()
    running_sum_sq += torch.sum(latent**2).item()

# Calculate global mean and variance
global_mean = running_sum / total_elements
global_variance = (running_sum_sq / total_elements) - (global_mean**2)

# Standard deviation is the square root of variance
global_std = global_variance**0.5
scale_factor = 1.0 / global_std

print(f"Total elements processed: {total_elements:,}")
print(f"Global Mean: {global_mean:.6f}")
print(f"Global Standard Deviation: {global_std:.6f}")
print(f"Recommended Scale Factor: {scale_factor:.6f}")

Processing latents: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1332/1332 [00:05<00:00, 258.69it/s]


Total elements processed: 2,793,406,464
Global Mean: -0.036366
Global Standard Deviation: 0.511053
Recommended Scale Factor: 1.956743


In [5]:
total_elements = 0
running_sum = 0.0
running_sum_sq = 0.0

for file_path in tqdm(val_files, desc="Processing latents"):
    # Load one file at a time
    latent = torch.load(file_path, map_location="cpu").to(torch.float32)

    # Accumulate element count, sum, and sum of squares
    total_elements += latent.numel()
    running_sum += torch.sum(latent).item()
    running_sum_sq += torch.sum(latent**2).item()

# Calculate global mean and variance
global_mean = running_sum / total_elements
global_variance = (running_sum_sq / total_elements) - (global_mean**2)

# Standard deviation is the square root of variance
global_std = global_variance**0.5
scale_factor = 1.0 / global_std

print(f"Total elements processed: {total_elements:,}")
print(f"Global Mean: {global_mean:.6f}")
print(f"Global Standard Deviation: {global_std:.6f}")
print(f"Recommended Scale Factor: {scale_factor:.6f}")

Processing latents: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:01<00:00, 265.67it/s]

Total elements processed: 589,299,712
Global Mean: -0.020564
Global Standard Deviation: 0.521808
Recommended Scale Factor: 1.916415


In [6]:
total_elements = 0
running_sum = 0.0
running_sum_sq = 0.0

for file_path in tqdm(latent_files, desc="Processing latents"):
    # Load one file at a time
    latent = torch.load(file_path, map_location="cpu").to(torch.float32)

    # Accumulate element count, sum, and sum of squares
    total_elements += latent.numel()
    running_sum += torch.sum(latent).item()
    running_sum_sq += torch.sum(latent**2).item()

# Calculate global mean and variance
global_mean = running_sum / total_elements
global_variance = (running_sum_sq / total_elements) - (global_mean**2)

# Standard deviation is the square root of variance
global_std = global_variance**0.5
scale_factor = 1.0 / global_std

print(f"Total elements processed: {total_elements:,}")
print(f"Global Mean: {global_mean:.6f}")
print(f"Global Standard Deviation: {global_std:.6f}")
print(f"Recommended Scale Factor: {scale_factor:.6f}")

Processing latents: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1613/1613 [00:02<00:00, 608.57it/s]

Total elements processed: 3,382,706,176
Global Mean: -0.033613
Global Standard Deviation: 0.512978
Recommended Scale Factor: 1.949401


Results for the train and validation datasets are close enough that there is no reason to pick one over the other, so the global scale factor will be used in Rectified Flow training.